In [0]:
# Databricks notebook source
# ============================================================================
# 07_universal_silver_processing
# Universal metadata-driven Silver pipeline with observed-time SCD2 + watermark
#
# Watermark design follows the existing Customers Silver notebook:
#   - Shared control Delta path: metadata/pipeline_watermark/
#   - One row per pipeline_name
#   - Null watermark means initial load (read all Bronze history)
#   - Incremental selection uses ingestion_timestamp >= watermark - 5 minutes
#   - Watermark advances only after Silver write and final validation succeed
#
# Important:
#   - ingestion_timestamp is Bronze ingestion/observation time, not source update time.
#   - The 5-minute overlap intentionally re-reads data. SCD2 version IDs make
#     already-committed versions idempotent; genuinely late/new conflicting
#     events still fail safely.
#   - SCD2 accepts one new distinct event per business key in a run. Multiple
#     new events for one key need a reliable source sequence/history design.
#   - Initial SCD2 bootstrap currently expects at most one event per key, matching
#     the inspected Google Ads Bronze snapshot. Historical multi-event bootstrap
#     needs a separately designed ordered-history process.
#   - No metadata config is activated or modified by this notebook.
# ============================================================================


In [0]:
# CELL 1 — IMPORTS
from datetime import datetime, timezone, timedelta
import uuid

from delta.tables import DeltaTable
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import StructType, TimestampType
from pyspark.storagelevel import StorageLevel


In [0]:
# CELL 2 — PARAMETERS, WIDGETS, PATHS, AND RUN CONFIGURATION
# Preserve existing widget values supplied by Databricks Jobs or interactive runs.
def ensure_text_widget(name, default, label):
    try:
        dbutils.widgets.get(name)
    except Exception:
        dbutils.widgets.text(name, default, label)


def ensure_dropdown_widget(name, default, choices, label):
    try:
        dbutils.widgets.get(name)
    except Exception:
        dbutils.widgets.dropdown(name, default, choices, label)


ensure_text_widget("dataset_name", "google_ads_campaigns", "Dataset Name")
ensure_dropdown_widget("full_load", "false", ["true", "false"], "Full Load")
ensure_dropdown_widget("reprocess", "false", ["true", "false"], "Reprocess")
ensure_dropdown_widget("require_ledger", "false", ["true", "false"], "Require Commit Ledger")

DATASET_NAME = dbutils.widgets.get("dataset_name").strip().lower()
FULL_LOAD = dbutils.widgets.get("full_load").strip().lower() == "true"
REPROCESS = dbutils.widgets.get("reprocess").strip().lower() == "true"
REQUIRE_LEDGER = dbutils.widgets.get("require_ledger").strip().lower() == "true"

if not DATASET_NAME:
    raise ValueError("dataset_name cannot be empty")
if REPROCESS and not FULL_LOAD:
    raise ValueError("reprocess=true requires full_load=true")

RUN_ID = str(uuid.uuid4())
RUN_TIMESTAMP = datetime.now(timezone.utc)

STORAGE_ACCOUNT = "nestlesalesdatalake"
CONTAINER = "customer-sales-datalake"
STORAGE_BASE_PATH = f"abfss://{CONTAINER}@{STORAGE_ACCOUNT}.dfs.core.windows.net"

BRONZE_BASE_PATH = f"{STORAGE_BASE_PATH}/bronze"
SILVER_BASE_PATH = f"{STORAGE_BASE_PATH}/silver"
QUARANTINE_BASE_PATH = f"{STORAGE_BASE_PATH}/quarantine/silver"
METADATA_BASE_PATH = f"{STORAGE_BASE_PATH}/metadata"

SILVER_DATASET_CONFIG_PATH = f"{METADATA_BASE_PATH}/silver_dataset_config/"
SILVER_TRANSFORM_CONFIG_PATH = f"{METADATA_BASE_PATH}/silver_column_transform_config/"
SILVER_DQ_CONFIG_PATH = f"{METADATA_BASE_PATH}/silver_dq_rules_config/"
SCHEMA_CONFIG_PATH = f"{METADATA_BASE_PATH}/schema_config/"

# Shared with the existing Customers Silver pipeline.
WATERMARK_CONTROL_PATH = f"{METADATA_BASE_PATH}/pipeline_watermark/"
WATERMARK_LOOKBACK_MINUTES = 5
PIPELINE_NAME = f"{DATASET_NAME}_silver"

TRANSFORMATION_NOTEBOOK_PATH = (
    "/Users/prachethanreddy01@gmail.com/customer-sales-project/"
    "03_silver/Metadata/05_silver_transformation_executor"
)
DQ_NOTEBOOK_PATH = (
    "/Users/prachethanreddy01@gmail.com/customer-sales-project/"
    "03_silver/Metadata/06_silver_dq_executor_corrected"
)

# Operational threshold; not currently part of the metadata table schema.
MAX_REJECT_RATIO = 0.05


print("UNIVERSAL SILVER CONFIGURATION")
print(f"Dataset                 : {DATASET_NAME}")
print(f"Pipeline name           : {PIPELINE_NAME}")
print(f"Run ID                  : {RUN_ID}")
print(f"Full load               : {FULL_LOAD}")
print(f"Reprocess               : {REPROCESS}")
print(f"Require ledger          : {REQUIRE_LEDGER}")
print(f"Watermark control path  : {WATERMARK_CONTROL_PATH}")
print(f"Watermark lookback      : {WATERMARK_LOOKBACK_MINUTES} minutes")
print(f"Bronze path             : {BRONZE_BASE_PATH}/{DATASET_NAME}/")
print(f"Silver path             : {SILVER_BASE_PATH}/{DATASET_NAME}/")
print(f"Quarantine path         : {QUARANTINE_BASE_PATH}/{DATASET_NAME}/")
print(f"Maximum reject ratio    : {MAX_REJECT_RATIO:.2%}")



UNIVERSAL SILVER CONFIGURATION
Dataset                 : google_ads_campaigns
Pipeline name           : google_ads_campaigns_silver
Run ID                  : 6eb0f458-03bb-4839-a75a-f68068f55e9f
Full load               : False
Reprocess               : False
Require ledger          : False
Watermark control path  : abfss://customer-sales-datalake@nestlesalesdatalake.dfs.core.windows.net/metadata/pipeline_watermark/
Watermark lookback      : 5 minutes
Bronze path             : abfss://customer-sales-datalake@nestlesalesdatalake.dfs.core.windows.net/bronze/google_ads_campaigns/
Silver path             : abfss://customer-sales-datalake@nestlesalesdatalake.dfs.core.windows.net/silver/google_ads_campaigns/
Quarantine path         : abfss://customer-sales-datalake@nestlesalesdatalake.dfs.core.windows.net/quarantine/silver/google_ads_campaigns/
Maximum reject ratio    : 5.00%


In [0]:
%run "/Users/prachethanreddy01@gmail.com/customer-sales-project/03_silver/Metadata/05_silver_transformation_executor"

In [0]:
%run "/Users/prachethanreddy01@gmail.com/customer-sales-project/03_silver/Metadata/06_silver_dq_executor_corrected"

Silver DQ Executor functions loaded successfully.
Supported DQ rules: ['NOT_NULL', 'UNIQUE', 'ACCEPTED_VALUES', 'REGEX', 'MIN_VALUE', 'MAX_VALUE', 'RANGE']
Supported severities: ['CRITICAL', 'HIGH', 'MEDIUM', 'LOW']
Supported failure actions: ['QUARANTINE', 'FAIL_DATASET', 'WARN']


In [0]:
# CELL 5 — COMMON HELPERS

def require_columns(df, required_columns, label):
    missing = [c for c in required_columns if c not in df.columns]
    if missing:
        raise ValueError(f"{label} is missing required columns: {missing}")


def quoted_col(column_name):
    return F.col("`" + column_name.replace("`", "``") + "`")


def require_delta_path(path, label):
    if not path or path.startswith("<"):
        raise ValueError(f"{label} path is not configured")
    if not DeltaTable.isDeltaTable(spark, path):
        raise ValueError(f"{label} Delta table/path does not exist: {path}")


def load_delta(path, label):
    require_delta_path(path, label)
    return spark.read.format("delta").load(path)


def dataset_paths(dataset_name):
    return {
        "bronze": f"{BRONZE_BASE_PATH}/{dataset_name}/",
        "silver": f"{SILVER_BASE_PATH}/{dataset_name}/",
        "quarantine": f"{QUARANTINE_BASE_PATH}/{dataset_name}/"
    }


def assert_unique_config_row(df, dataset_name, label):
    rows = df.filter(F.lower(F.trim(F.col("dataset_name"))) == dataset_name).limit(2).collect()
    if len(rows) != 1:
        raise ValueError(
            f"Expected exactly one {label} row for {dataset_name}; found {len(rows)}"
        )
    return rows[0].asDict(recursive=True)


In [0]:
# CELL 6 — VERIFY ALL FOUR METADATA DELTA TABLES

def verify_metadata_tables():
    metadata_paths = {
        "silver_dataset_config": SILVER_DATASET_CONFIG_PATH,
        "silver_column_transform_config": SILVER_TRANSFORM_CONFIG_PATH,
        "silver_dq_rules_config": SILVER_DQ_CONFIG_PATH,
        "schema_config": SCHEMA_CONFIG_PATH
    }
    verification = []
    for table_name, path in metadata_paths.items():
        exists = DeltaTable.isDeltaTable(spark, path)
        verification.append((table_name, path, exists))
        print(f"{table_name}: {'EXISTS' if exists else 'NOT FOUND'} | {path}")
    missing = [name for name, _, exists in verification if not exists]
    if missing:
        raise ValueError(f"Required metadata Delta tables are missing: {missing}")
    return verification


In [0]:
# CELL 7 — LOAD AND VALIDATE DATASET CONFIGURATION

def load_dataset_config(dataset_config_df, dataset_name):
    required = [
        "dataset_name", "source_system", "is_active", "config_status",
        "load_strategy", "scd_type", "business_key_columns",
        "sequence_column", "deduplication_enabled", "partition_columns",
        "scd2_tracked_columns", "effective_from_column",
        "effective_to_column", "is_current_column"
    ]
    require_columns(dataset_config_df, required, "silver_dataset_config")
    config = assert_unique_config_row(dataset_config_df, dataset_name, "dataset config")

    if config["is_active"] is not True:
        raise ValueError(f"Dataset config for {dataset_name} is inactive")
    if str(config["config_status"]).upper() != "APPROVED":
        raise ValueError(
            f"Dataset config for {dataset_name} is {config['config_status']}, not APPROVED"
        )

    keys = config.get("business_key_columns") or []
    if not keys:
        raise ValueError("business_key_columns cannot be empty")

    scd_type = int(config["scd_type"])
    if scd_type not in (1, 2):
        raise ValueError(f"Unsupported scd_type: {scd_type}")

    load_strategy = str(config["load_strategy"]).upper()
    if load_strategy not in ("FULL", "INCREMENTAL"):
        raise ValueError(f"Unsupported load_strategy: {load_strategy}")

    if scd_type == 2:
        if not config.get("sequence_column"):
            raise ValueError(
                "SCD2 requires sequence_column. For this dataset configure "
                "ingestion_timestamp for observed-time SCD2."
            )
        if str(config["sequence_column"]).lower() != "ingestion_timestamp":
            raise ValueError(
                "This implementation is OBSERVED_TIME SCD2 and requires "
                "sequence_column='ingestion_timestamp'."
            )
        if not config.get("scd2_tracked_columns"):
            raise ValueError("SCD2 requires scd2_tracked_columns")
        if not (config.get("effective_from_column") 
                # and config.get("effective_to_column")
                and config.get("is_current_column")):
            raise ValueError("SCD2 effective-date/current-flag columns are not configured")

    return config


In [0]:
# CELL 8 — LOAD TRANSFORMATION AND DQ METADATA

def load_transformation_rules(transform_config_df, dataset_name):
    required = [
        "dataset_name", "source_column", "target_column", "transformation_type",
        "transformation_parameters", "execution_order", "is_active", "description"
    ]
    require_columns(transform_config_df, required, "silver_column_transform_config")
    rows = (
        transform_config_df
        .filter(
            (F.lower(F.trim(F.col("dataset_name"))) == dataset_name) &
            (F.col("is_active") == F.lit(True))
        )
        .orderBy("execution_order", "source_column", "transformation_type")
        .collect()
    )
    return [r.asDict(recursive=True) for r in rows]


def load_approved_dq_rules(dq_config_df, dataset_name):
    required = [
        "dataset_name", "rule_id", "column_name", "rule_type", "rule_parameters",
        "severity", "failure_action", "execution_order", "is_active",
        "rule_status", "description"
    ]
    require_columns(dq_config_df, required, "silver_dq_rules_config")
    rows = (
        dq_config_df
        .filter(
            (F.lower(F.trim(F.col("dataset_name"))) == dataset_name) &
            (F.col("is_active") == F.lit(True)) &
            (F.upper(F.col("rule_status")) == F.lit("APPROVED"))
        )
        .orderBy("execution_order", "rule_id")
        .collect()
    )
    return [r.asDict(recursive=True) for r in rows]


def load_all_metadata(dataset_name):
    dataset_df = load_delta(SILVER_DATASET_CONFIG_PATH, "silver_dataset_config")
    transform_df = load_delta(SILVER_TRANSFORM_CONFIG_PATH, "silver_column_transform_config")
    dq_df = load_delta(SILVER_DQ_CONFIG_PATH, "silver_dq_rules_config")
    return {
        "dataset_config": load_dataset_config(dataset_df, dataset_name),
        "transformation_rules": load_transformation_rules(transform_df, dataset_name),
        "dq_rules": load_approved_dq_rules(dq_df, dataset_name)
    }
#print(load_all_metadata("google_ads_campaigns"))


In [0]:
load_all_metadata("google_ads_campaigns")

{'dataset_config': {'dataset_name': 'google_ads_campaigns',
  'source_system': 'google_ads',
  'is_active': True,
  'config_status': 'APPROVED',
  'load_strategy': 'INCREMENTAL',
  'scd_type': '2',
  'business_key_columns': ['customer_id', 'campaign_id'],
  'sequence_column': 'ingestion_timestamp',
  'deduplication_enabled': True,
  'partition_columns': [],
  'scd2_tracked_columns': ['campaign_name',
   'campaign_status',
   'advertising_channel_type',
   'advertising_channel_sub_type',
   'campaign_budget_resource_name',
   'bidding_strategy_type',
   'start_date',
   'end_date',
   'campaign_primary_status',
   'serving_status',
   'target_country',
   'business_brand',
   'product_category',
   'business_objective'],
  'effective_from_column': '2026-01-01',
  'effective_to_column': 'effective_to',
  'is_current_column': 'true',
  'created_at': datetime.datetime(2026, 11, 16, 16, 11, 32, 530102),
  'updated_at': datetime.datetime(2026, 11, 16, 16, 11, 32, 530102)},
 'transformation_r

In [0]:
# CELL 9 — LOAD ACTIVE SOURCE SCHEMA

def load_schema_ddl(schema_config_path, dataset_name):
    schema_df = load_delta(schema_config_path, "schema_config")
    require_columns(
        schema_df,
        ["dataset_name", "source_system", "schema_ddl", "schema_version", "is_active"],
        "schema_config"
    )
    rows = (
        schema_df
        .filter(
            (F.lower(F.trim(F.col("dataset_name"))) == dataset_name) &
            (F.col("is_active") == F.lit(True))
        )
        .limit(2)
        .collect()
    )
    if len(rows) != 1:
        raise ValueError(
            f"Expected exactly one active schema for {dataset_name}; found {len(rows)}"
        )
    row = rows[0]
    if not row["schema_ddl"] or not row["schema_ddl"].strip():
        raise ValueError(f"schema_ddl is empty for {dataset_name}")
    return row["schema_ddl"], row["source_system"], row["schema_version"]

schema_ddl, source_system, schema_version = load_schema_ddl(
    SCHEMA_CONFIG_PATH,
    "google_ads_campaigns"
)
# print(schema_ddl)
# print(source_system)
# print(schema_version)

In [0]:
# CELL 10 — SCHEMA DRIFT VALIDATION

# The existing schema_config defines source/business columns only. Bronze
# ingestion adds its own technical metadata columns, which must not be treated
# as unexpected source-schema drift.
def check_schema_drift(bronze_df, expected_source_columns, mode="STRICT"):
    mode = str(mode).upper()
    if mode not in ("STRICT", "ADDITIVE", "APPROVAL_REQUIRED"):
        raise ValueError(f"Unsupported schema drift mode: {mode}")

    actual_source_columns = [c for c in bronze_df.columns if c not in BRONZE_LINEAGE_COLUMNS]
    missing = sorted(set(expected_source_columns) - set(actual_source_columns))
    added = sorted(set(actual_source_columns) - set(expected_source_columns))
    has_drift = bool(missing or added)

    if mode == "STRICT" and has_drift:
        raise ValueError(f"STRICT schema drift. Missing={missing}; Added={added}")
    if mode == "ADDITIVE" and missing:
        raise ValueError(f"Missing expected source columns: {missing}")
    if mode == "APPROVAL_REQUIRED" and has_drift:
        raise ValueError(f"Schema approval required. Missing={missing}; Added={added}")

    result = {"mode": mode, "missing": missing, "added": added, "has_drift": has_drift}
    print(f"Schema drift check: {result}")
    return result


In [0]:
# CELL 11 — SHARED WATERMARK CONTROL TABLE

WATERMARK_SCHEMA = StructType.fromDDL("""
    pipeline_name STRING,
    last_processed_timestamp TIMESTAMP,
    last_run_id STRING,
    updated_at TIMESTAMP
""")


def ensure_watermark_table():                               
    """Create the shared Delta control table if it does not exist."""
    if not DeltaTable.isDeltaTable(spark, WATERMARK_CONTROL_PATH):
        (spark.createDataFrame([], WATERMARK_SCHEMA)
             .write.format("delta").mode("overwrite").save(WATERMARK_CONTROL_PATH))

    control_df = spark.read.format("delta").load(WATERMARK_CONTROL_PATH)
    required = ["pipeline_name", "last_processed_timestamp", "last_run_id", "updated_at"]
    require_columns(control_df, required, "pipeline_watermark")
    duplicate_pipeline = (control_df.groupBy("pipeline_name").count()
                          .filter(F.col("count") > 1).limit(1).count())
    if duplicate_pipeline:
        raise ValueError("pipeline_watermark contains duplicate pipeline_name rows")

    if control_df.filter(F.col("pipeline_name") == PIPELINE_NAME).limit(1).count() == 0:
        initial_row = [(PIPELINE_NAME, None, None, None)]
        (spark.createDataFrame(initial_row, WATERMARK_SCHEMA)
             .write.format("delta").mode("append").save(WATERMARK_CONTROL_PATH))

    return spark.read.format("delta").load(WATERMARK_CONTROL_PATH)


def read_last_watermark():
    control_df = ensure_watermark_table()
    row = (control_df.filter(F.col("pipeline_name") == PIPELINE_NAME)
           .select("last_processed_timestamp").first())
    return row["last_processed_timestamp"]


def update_watermark(processed_timestamp):
    """Commit watermark only after Silver write and final validation succeed."""
    if processed_timestamp is None:
        print("Watermark unchanged: no processed ingestion timestamp was found")
        return

    update_df = spark.createDataFrame(
        [(PIPELINE_NAME, processed_timestamp, RUN_ID, RUN_TIMESTAMP.replace(tzinfo=None))],
        WATERMARK_SCHEMA
    )
    (DeltaTable.forPath(spark, WATERMARK_CONTROL_PATH).alias("target")
        .merge(update_df.alias("source"), "target.pipeline_name = source.pipeline_name")
        .whenMatchedUpdate(set={
            "last_processed_timestamp": "greatest(target.last_processed_timestamp, source.last_processed_timestamp)",
            "last_run_id": "source.last_run_id",
            "updated_at": "source.updated_at"
        })
        .whenNotMatchedInsertAll()
        .execute())

    print(f"Watermark committed through: {processed_timestamp}")


In [0]:
# CELL 12 — READ BRONZE USING WATERMARK + LOOKBACK WINDOW
BRONZE_LINEAGE_COLUMNS = [
    "source_file", "source_system", "run_id", "batch_id", "ingestion_timestamp"
]


def read_bronze(bronze_path, last_watermark, full_load=False, require_ledger=False):
    require_delta_path(bronze_path, "Bronze dataset")
    bronze_df = spark.read.format("delta").load(bronze_path)
    require_columns(bronze_df, BRONZE_LINEAGE_COLUMNS, "Bronze dataset")

    if require_ledger:
        raise NotImplementedError(
            "Commit-ledger filtering is not configured. Keep require_ledger=false "
            "until the ledger path/schema and successful-commit contract are defined."
        )

    if not isinstance(bronze_df.schema["ingestion_timestamp"].dataType, TimestampType):
        raise TypeError("Bronze ingestion_timestamp must be TIMESTAMP")

    if full_load:
        selected = bronze_df
        print("Bronze selection: all available records (full_load=true)")
    elif last_watermark is None:
        selected = bronze_df
        print("Bronze selection: INITIAL LOAD — all available Bronze records")
    else:
        boundary = last_watermark - timedelta(minutes=WATERMARK_LOOKBACK_MINUTES)
        selected = bronze_df.filter(F.col("ingestion_timestamp") >= F.lit(boundary))
        print(f"Bronze selection: ingestion_timestamp >= {boundary} (watermark overlap)")

    selected_count = selected.count()
    file_count = selected.select("source_file").distinct().count()
    print(f"Selected Bronze rows={selected_count}; source files={file_count}")
    return selected, file_count


In [0]:
# CELL 12 — SAFE TYPE STANDARDIZATION

def standardize_bronze(bronze_df, schema_ddl):
    expected_schema = StructType.fromDDL(schema_ddl)
    result_df = bronze_df
    cast_error_expressions = []

    for field in expected_schema.fields:
        column_name = field.name
        if column_name not in result_df.columns:
            raise ValueError(f"Expected source column missing before casting: {column_name}")
        target_type = field.dataType.simpleString()
        if target_type == "string":
            continue

        original = quoted_col(column_name)
        converted = F.expr(
            f"try_cast(`{column_name.replace('`', '``')}` AS {target_type})"
        )
        cast_error_expressions.append(
            F.when(original.isNotNull() & converted.isNull(), F.lit(column_name))
        )
        result_df = result_df.withColumn(column_name, converted)

    if cast_error_expressions:
        result_df = result_df.withColumn(
            "_cast_errors", F.concat_ws(",", *cast_error_expressions)
        )
    else:
        result_df = result_df.withColumn("_cast_errors", F.lit(""))

    return result_df


In [0]:
def apply_dataset_transformations(df, transformation_rules):

    if not transformation_rules:
        print("No active transformation rules.")
        return df

    print("Applying", len(transformation_rules), "transformation rules")

    return apply_silver_transformations(
        df=df,
        transformation_rules=transformation_rules
    )

In [0]:
# CELL 14 — EXECUTE NOTEBOOK 06 DQ RULES AND CAST-FAILURE QUARANTINE

def execute_dataset_dq(df, dq_rules):
    if dq_rules:
        dq_result = execute_dq_rules(df, dq_rules)
        passed_df = dq_result["passed_df"]
        quarantined_df = dq_result["quarantined_df"]
        rule_metrics = dq_result["rule_results"]
    else:
        passed_df = df
        quarantined_df = df.limit(0)
        rule_metrics = []
                                
    # The DQ executor quarantines only QUARANTINE-action failures. WARN rows
    # remain in passed_df; FAIL_DATASET raises inside the executor.   
    if "_cast_errors" in df.columns:
        cast_failed = df.filter(F.col("_cast_errors") != "")
        passed_df = passed_df.filter(
            F.col("_cast_errors").isNull() | (F.col("_cast_errors") == "")
        )
        quarantined_df = quarantined_df.unionByName(
            cast_failed, allowMissingColumns=True
        )

    quarantined_df = quarantined_df.dropDuplicates()
    return passed_df, quarantined_df, rule_metrics


In [0]:
# CELL 15 — DETERMINISTIC DEDUPLICATION

def deduplicate_records(df, business_keys, sequence_column=None):
    require_columns(df, business_keys, "Silver input")
    if not business_keys:
        raise ValueError("At least one business key is required")

    if sequence_column:
        require_columns(df, [sequence_column], "Silver input sequence")
        primary_order = quoted_col(sequence_column).desc_nulls_last()
    else:
        require_columns(df, ["ingestion_timestamp"], "Bronze lineage")
        primary_order = F.col("ingestion_timestamp").desc_nulls_last()

    tie_breakers = [
        quoted_col(c).desc_nulls_last()
        for c in ("run_id", "source_file", "batch_id")
        if c in df.columns
    ]
    window = Window.partitionBy(*[quoted_col(k) for k in business_keys]).orderBy(
        primary_order, *tie_breakers
    )
    return (
        df.withColumn("_silver_dedup_rank", F.row_number().over(window))
          .filter(F.col("_silver_dedup_rank") == 1)
          .drop("_silver_dedup_rank")
    )


In [0]:
# CELL 16 — ROW HASH AND SILVER TECHNICAL COLUMNS

def add_row_hash(df, hash_columns):
    require_columns(df, hash_columns, "Row hash input")
    return df.withColumn(
        "_silver_row_hash",
        F.sha2(
            F.to_json(F.struct(*[quoted_col(c).alias(c) for c in hash_columns])),
            256
        )
    )


def prepare_silver_source(df, business_keys):
    # Bronze operational metadata is retained in Silver for lineage. Internal
    # processing columns are not written to the business Silver dataset.
    internal_columns = [
        c for c in df.columns
        if c.startswith("_cast_") or c.startswith("_silver_dq_")
    ]
    result = df.drop(*internal_columns)
    result = result.withColumn("created_at", F.current_timestamp())
    result = result.withColumn("updated_at", F.current_timestamp())
    return result


In [0]:
# CELL 17 — IDEMPOTENT QUARANTINE WRITE

def write_quarantine(rejected_df, quarantine_path, dataset_name):
    if rejected_df.limit(1).count() == 0:
        print("No rejected records to quarantine")
        return 0

    output = (rejected_df
        .withColumn("_quarantine_dataset", F.lit(dataset_name))
        .withColumn("_quarantine_run_id", F.lit(RUN_ID))
        .withColumn("_quarantine_timestamp", F.current_timestamp())
        .withColumn("_quarantine_reason", F.lit("DQ_OR_TYPE_CAST_FAILURE")))

    # Hash only stable source content + dataset + reason. Exclude run ID and
    # quarantine timestamp so retrying the same rejected row remains idempotent.
    stable_hash_columns = rejected_df.columns + ["_quarantine_dataset", "_quarantine_reason"]
    output = (output.withColumn(
        "_quarantine_record_hash",
        F.sha2(F.to_json(F.struct(*[quoted_col(c).alias(c) for c in stable_hash_columns])), 256)
    ).dropDuplicates(["_quarantine_record_hash"]))

    if not DeltaTable.isDeltaTable(spark, quarantine_path):
        output.write.format("delta").mode("overwrite").save(quarantine_path)
        return output.count()

    (DeltaTable.forPath(spark, quarantine_path).alias("target")
        .merge(output.alias("source"),
               "target._quarantine_record_hash = source._quarantine_record_hash")
        .whenNotMatchedInsertAll().execute())
    return output.count()


In [0]:
# CELL 18 — SCD1 INCREMENTAL MERGE / FULL REPROCESS

def merge_scd1(source_df, silver_path, business_keys, reprocess=False):
    hash_columns = [
        c for c in source_df.columns
        if c not in ("created_at", "updated_at")
        and c != "_silver_row_hash"
    ]
    source = add_row_hash(source_df, hash_columns)

    if reprocess:
        (source
         .withColumn("created_at", F.current_timestamp())
         .withColumn("updated_at", F.current_timestamp())
         .write.format("delta").mode("overwrite")
         .option("overwriteSchema", "true")
         .save(silver_path))
        return source.count()

    if not DeltaTable.isDeltaTable(spark, silver_path):
        (source.write.format("delta").mode("overwrite").save(silver_path))
        return source.count()

    target_df = spark.read.format("delta").load(silver_path)
    if set(target_df.columns) != set(source.columns):
        raise ValueError(
            "Existing Silver schema differs from incoming source schema. "
            "Perform an approved schema migration or explicit reprocess. "
            f"Target-only={sorted(set(target_df.columns)-set(source.columns))}; "
            f"Source-only={sorted(set(source.columns)-set(target_df.columns))}"
        )

    merge_condition = " AND ".join(
        [f"t.`{k.replace('`', '``')}` = s.`{k.replace('`', '``')}`" for k in business_keys]
    )
    update_set = {
        c: f"s.`{c.replace('`', '``')}`"
        for c in source.columns
        if c not in ("created_at",)
    }
    update_set["updated_at"] = "current_timestamp()"
    insert_values = {
        c: f"s.`{c.replace('`', '``')}`"
        for c in source.columns
    }

    (DeltaTable.forPath(spark, silver_path).alias("t")
        .merge(source.alias("s"), merge_condition)
        .whenMatchedUpdate(
            condition="NOT (t._silver_row_hash <=> s._silver_row_hash)",
            set=update_set
        )
        .whenNotMatchedInsert(values=insert_values)
        .execute())
    return source.count()


In [0]:

# CELL 19 — JOIN-BASED OBSERVED-TIME SCD2 WITH OVERLAP IDEMPOTENCY

def merge_scd2_incremental(source_df, silver_path, config):

    """Apply SCD2 using LEFT ANTI, INNER JOIN, and UNION BY NAME."""

    # ---------------------------------------------------------
    # 1. Prevent unsupported full load / reprocess
    # ---------------------------------------------------------

    if FULL_LOAD or REPROCESS:
        raise ValueError(
            "SCD2 full_load/reprocess is disabled in this incremental notebook. "
            "Use a separately approved historical bootstrap/rebuild workflow."
        )

    # ---------------------------------------------------------
    # 2. Read configuration
    # ---------------------------------------------------------

    keys = config["business_key_columns"]
    tracked = config["scd2_tracked_columns"]
    sequence = config["sequence_column"]
    effective_from = config["effective_from_column"]
    effective_to = config["effective_to_column"]
    current_flag = config["is_current_column"]

    require_columns(
        source_df,
        keys + tracked + [sequence],
        "SCD2 source"
    )

    # ---------------------------------------------------------
    # 3. Validate sequence and business keys
    # ---------------------------------------------------------

    if not isinstance(source_df.schema[sequence].dataType, TimestampType):
        raise TypeError(
            f"SCD2 sequence column {sequence} must be TIMESTAMP"
        )

    if source_df.filter(
        quoted_col(sequence).isNull()
    ).limit(1).count():
        raise ValueError(
            f"SCD2 source contains null {sequence} values"
        )

    null_key_condition = None

    for k in keys:
        condition = quoted_col(k).isNull()
        null_key_condition = (
            condition if null_key_condition is None
            else null_key_condition | condition
        )

    if source_df.filter(null_key_condition).limit(1).count():
        raise ValueError("SCD2 source contains null business-key values")

    # ---------------------------------------------------------
    # 4. Generate row hash and version ID
    # ---------------------------------------------------------

    hashed = add_row_hash(source_df, tracked)

    version_id_columns = keys + [
        sequence,
        "_silver_row_hash"
    ]

    hashed = hashed.withColumn(
        "_scd2_version_id",
        F.sha2(
            F.to_json(
                F.struct(*[
                    quoted_col(c).alias(c)
                    for c in version_id_columns
                ])
            ),
            256
        )
    )

    # ---------------------------------------------------------
    # 5. Detect conflicting events at the same timestamp
    # ---------------------------------------------------------

    same_time_conflicts = (
        hashed
        .groupBy(*(keys + [sequence]))
        .agg(
            F.countDistinct("_silver_row_hash").alias("state_count")
        )
        .filter(F.col("state_count") > 1)
        .limit(1)
        .count()
    )

    if same_time_conflicts:
        raise ValueError(
            "Conflicting SCD2 states share the same business key "
            "and ingestion_timestamp"
        )

    # Remove exact repeated copies of the same event
    incoming = hashed.dropDuplicates(
        keys + [sequence, "_silver_row_hash"]
    )

    # ---------------------------------------------------------
    # 6. Initial SCD2 bootstrap
    # ---------------------------------------------------------

    if not DeltaTable.isDeltaTable(spark, silver_path):

        multiple_events = (
            incoming
            .select(*(keys + [sequence]))
            .dropDuplicates()
            .groupBy(*keys)
            .count()
            .filter(F.col("count") > 1)
            .limit(1)
            .count()
        )

        if multiple_events:
            raise ValueError(
                "Initial SCD2 bootstrap found multiple events for "
                "a business key. A chronological history bootstrap "
                "is required."
            )

        initial = (
            incoming
            .withColumn(effective_from, quoted_col(sequence))
            .withColumn(
                effective_to,
                F.lit(None).cast("timestamp")
            )
            .withColumn(current_flag, F.lit(True))
            .withColumn("created_at", F.current_timestamp())
            .withColumn("updated_at", F.current_timestamp())
        )

        initial.write.format("delta").mode("ignore").save(silver_path)

        return initial.count()

    # ---------------------------------------------------------
    # 7. Read existing Silver target
    # ---------------------------------------------------------

    target_df = spark.read.format("delta").load(silver_path)

    require_columns(
        target_df,
        keys + [
            effective_from,
            effective_to,
            current_flag,
            "_scd2_version_id",
            "_silver_row_hash"
        ],
        "Existing SCD2 Silver target"
    )

    unexpected = sorted(
        set(incoming.columns) - set(target_df.columns)
    )

    if unexpected:
        raise ValueError(
            f"Incoming SCD2 columns are missing from target schema: {unexpected}"
        )

    # ---------------------------------------------------------
    # 8. Remove already-committed events (overlap protection)
    # ---------------------------------------------------------

    known_versions = (
        target_df
        .select("_scd2_version_id")
        .distinct()
    )

    new_events = incoming.join(
        known_versions,
        "_scd2_version_id",
        "left_anti"
    )

    if new_events.limit(1).count() == 0:
        print(
            "SCD2: overlap contained only previously committed versions; no-op"
        )
        return 0

    # ---------------------------------------------------------
    # 9. Allow only one new event per business key per run
    # ---------------------------------------------------------

    multiple_new_events = (
        new_events
        .select(*(keys + [sequence]))
        .dropDuplicates()
        .groupBy(*keys)
        .count()
        .filter(F.col("count") > 1)
        .limit(1)
        .count()
    )

    if multiple_new_events:
        raise ValueError(
            "Multiple new ingestion events for one business key were selected. "
            "The current SCD2 design requires one new event per key per run."
        )

    # ---------------------------------------------------------
    # 10. Get current Silver records
    # ---------------------------------------------------------

    current_target = target_df.filter(
        quoted_col(current_flag) == F.lit(True)
    )

    # Build business-key matching condition for aliased DataFrames
    key_condition = None

    for k in keys:
        escaped = k.replace("`", "``")

        condition = (
            F.col(f"s.`{escaped}`") ==
            F.col(f"t.`{escaped}`")
        )

        key_condition = (
            condition if key_condition is None
            else key_condition & condition
        )

    # ---------------------------------------------------------
    # 11. NEW RECORDS — LEFT ANTI JOIN
    # ---------------------------------------------------------

    new_records = new_events.join(
        current_target.select(*keys),
        keys,
        "left_anti"
    )

    # ---------------------------------------------------------
    # 12. MATCHED RECORDS — INNER JOIN
    # ---------------------------------------------------------

    matched_records = (
        new_events.alias("s")
        .join(
            current_target.alias("t"),
            key_condition,
            "inner"
        )
    )

    # ---------------------------------------------------------
    # 13. Detect late-arriving events
    # ---------------------------------------------------------

    late_condition = (
        F.col(f"t.`{effective_from}`").isNotNull()
        &
        (
            F.col(f"s.`{sequence}`") <
            F.col(f"t.`{effective_from}`")
        )
    )

    if matched_records.filter(
        late_condition
    ).limit(1).count():
        raise ValueError(
            "Late-arriving SCD2 event detected; manual reconciliation is required"
        )

    # ---------------------------------------------------------
    # 14. Detect same-time conflicting states
    # ---------------------------------------------------------

    hash_changed = ~F.col(
        "s.`_silver_row_hash`"
    ).eqNullSafe(
        F.col("t.`_silver_row_hash`")
    )

    same_time_condition = (
        F.col(f"t.`{effective_from}`").isNotNull()
        &
        (
            F.col(f"s.`{sequence}`") ==
            F.col(f"t.`{effective_from}`")
        )
        &
        hash_changed
    )

    if matched_records.filter(
        same_time_condition
    ).limit(1).count():
        raise ValueError(
            "Same-time conflicting SCD2 state detected; "
            "refusing ambiguous ordering"
        )

    # ---------------------------------------------------------
    # 15. Build tracked-column change condition
    # ---------------------------------------------------------

    changed_condition = None

    for c in tracked:

        escaped = c.replace("`", "``")

        column_changed = ~F.col(
            f"s.`{escaped}`"
        ).eqNullSafe(
            F.col(f"t.`{escaped}`")
        )

        changed_condition = (
            column_changed if changed_condition is None
            else changed_condition | column_changed
        )

    # ---------------------------------------------------------
    # 16. CHANGED RECORDS — INNER JOIN + COLUMN COMPARISON
    # ---------------------------------------------------------

    changed_records = (
        matched_records
        .filter(changed_condition)
        .select("s.*")
    )

    # ---------------------------------------------------------
    # 17. UNCHANGED RECORDS — MATCHED + NO COLUMN CHANGE
    # ---------------------------------------------------------

    unchanged_records = (
        matched_records
        .filter(~changed_condition)
        .select("s.*")
    )

    # Unchanged records are deliberately not written again.

    # ---------------------------------------------------------
    # 18. Combine NEW + CHANGED records using UNION BY NAME
    # ---------------------------------------------------------

    records_to_insert = new_records.unionByName(
        changed_records
    )

    if records_to_insert.limit(1).count() == 0:
        print(
            "SCD2: all new events match current state; no version change required"
        )
        return 0

    # ---------------------------------------------------------
    # 19. Prepare CLOSE actions for CHANGED records only
    # ---------------------------------------------------------

    close_actions = (
        changed_records
        .select(
            *[quoted_col(k) for k in keys],
            quoted_col(sequence)
        )
        .withColumn("_scd2_action", F.lit("CLOSE"))
        .withColumn(
            "_scd2_version_id",
            F.lit(None).cast("string")
        )
    )

    # ---------------------------------------------------------
    # 20. Prepare INSERT actions for NEW + CHANGED records
    # ---------------------------------------------------------

    insert_actions = (
        records_to_insert
        .withColumn(effective_from, quoted_col(sequence))
        .withColumn(
            effective_to,
            F.lit(None).cast("timestamp")
        )
        .withColumn(current_flag, F.lit(True))
        .withColumn("created_at", F.current_timestamp())
        .withColumn("updated_at", F.current_timestamp())
        .withColumn("_scd2_action", F.lit("INSERT"))
    )

    # ---------------------------------------------------------
    # 21. Align action DataFrames to existing Silver schema
    # ---------------------------------------------------------

    target_columns = target_df.columns

    def align_to_target(df):

        result = df

        for name in target_columns:

            if name not in result.columns:

                result = result.withColumn(
                    name,
                    F.lit(None).cast(
                        target_df.schema[name].dataType
                    )
                )

        return result.select(
            *target_columns,
            "_scd2_action"
        )

    actions = (
        align_to_target(close_actions)
        .unionByName(
            align_to_target(insert_actions)
        )
    )

    # ---------------------------------------------------------
    # 22. Build Delta MERGE conditions
    # ---------------------------------------------------------

    key_match = " AND ".join(
        [
            f"target.`{k.replace('`', '``')}` = "
            f"source.`{k.replace('`', '``')}`"
            for k in keys
        ]
    )

    merge_condition = (
        f"(source._scd2_action = 'CLOSE' "
        f"AND {key_match} "
        f"AND target.`{current_flag}` = true) "
        "OR "
        "(source._scd2_action = 'INSERT' "
        "AND target._scd2_version_id = source._scd2_version_id)"
    )

    # ---------------------------------------------------------
    # 23. Define CLOSE update values
    # ---------------------------------------------------------

    update_values = {
        effective_to: f"source.`{sequence}`",
        current_flag: "false",
        "updated_at": "current_timestamp()"
    }

    # ---------------------------------------------------------
    # 24. Define INSERT values
    # ---------------------------------------------------------

    insert_values = {
        c: f"source.`{c}`"
        for c in target_columns
    }

    # ---------------------------------------------------------
    # 25. Execute one Delta MERGE for CLOSE + INSERT
    # ---------------------------------------------------------

    (
        DeltaTable.forPath(spark, silver_path)
        .alias("target")
        .merge(
            actions.alias("source"),
            merge_condition
        )
        .whenMatchedUpdate(
            condition=(
                f"source._scd2_action = 'CLOSE' "
                f"AND target.`{current_flag}` = true"
            ),
            set=update_values
        )
        .whenNotMatchedInsert(
            condition="source._scd2_action = 'INSERT'",
            values=insert_values
        )
        .execute()
    )

    return records_to_insert.count()


In [0]:
# CELL 20 — POST-WRITE VALIDATION

def validate_silver(silver_path, config):
    silver_df = spark.read.format("delta").load(silver_path)
    keys = config["business_key_columns"]
    scd_type = int(config["scd_type"])

    if scd_type == 1:
        duplicate_keys = (
            silver_df.groupBy(*keys).count()
            .filter(F.col("count") > 1).limit(1).count()
        )                                                  
        if duplicate_keys:
            raise ValueError("SCD1 validation failed: duplicate business keys found")
    else:
        current_col = config["is_current_column"]
        duplicate_current = (
            silver_df.filter(F.col(current_col) == F.lit(True))
            .groupBy(*keys).count().filter(F.col("count") > 1).limit(1).count()
        )
        if duplicate_current:
            raise ValueError("SCD2 validation failed: multiple current rows per key")

        start_col = config["effective_from_column"]
        end_col = config["effective_to_column"]
        invalid_intervals = silver_df.filter(
            quoted_col(start_col).isNull() |
            (F.col(end_col).isNotNull() & (quoted_col(end_col) <= quoted_col(start_col)))
        ).limit(1).count()
        if invalid_intervals:
            raise ValueError("SCD2 validation failed: invalid effective-date interval")

        invalid_current_end = silver_df.filter(
            (quoted_col(current_col) == F.lit(True)) & quoted_col(end_col).isNotNull()
        ).limit(1).count()
        if invalid_current_end:
            raise ValueError("SCD2 validation failed: current row has non-null effective_to")

        duplicate_versions = (
            silver_df.groupBy(*(keys + [start_col])).count()
            .filter(F.col("count") > 1).limit(1).count()
        )
        if duplicate_versions:
            raise ValueError("SCD2 validation failed: duplicate key/effective_from versions")

    return silver_df


In [0]:

# CELL 21 — MAIN PIPELINE ORCHESTRATION

def run_silver_pipeline():

    try:
        print(
            f"Starting Silver pipeline | dataset={DATASET_NAME} | run={RUN_ID}"
        )

        verify_metadata_tables()

        metadata = load_all_metadata(DATASET_NAME)

        config = metadata["dataset_config"]
        transform_rules = metadata["transformation_rules"]
        dq_rules = metadata["dq_rules"]

        paths = dataset_paths(DATASET_NAME)

        # Prevent unsupported SCD2 full rebuild operations.
        if int(config["scd_type"]) == 2 and (FULL_LOAD or REPROCESS):
            raise ValueError(
                "SCD2 full_load/reprocess is disabled. Use the approved initial "
                "watermark bootstrap or a separate historical rebuild workflow."
            )

        # Load and validate source schema.
        schema_ddl, schema_source_system, schema_version = load_schema_ddl(
            SCHEMA_CONFIG_PATH,
            DATASET_NAME
        )       

        expected_schema = StructType.fromDDL(schema_ddl)

        expected_source_columns = [
            field.name for field in expected_schema.fields
        ]

        if config.get("source_system") and schema_source_system:
            if config["source_system"].lower() != schema_source_system.lower():
                raise ValueError(
                    "source_system mismatch between dataset config "
                    "and active schema_config"
                )

        # Read the previous successful watermark.
        last_watermark = read_last_watermark()

        print(f"Previous successful watermark: {last_watermark}")

        # Select Bronze records using watermark and lookback logic.
        bronze_df, file_count = read_bronze(
            paths["bronze"],
            last_watermark,
            FULL_LOAD,
            REQUIRE_LEDGER
        )

        if bronze_df.limit(1).count() == 0:
            print(
                "No Bronze records in the selected watermark window. "
                "Nothing to process."
            )
            return None

        # Schema validation.
        check_schema_drift(
            bronze_df,
            expected_source_columns,
            mode="STRICT"
        )

        # Standardize and transform Bronze records.
        standardized_df = standardize_bronze(
            bronze_df,
            schema_ddl
        )

        transformed_df = apply_dataset_transformations(
            standardized_df,
            transform_rules
        )

        require_columns(
            transformed_df,
            config["business_key_columns"],
            "Transformed Silver input"
        )

        # Execute data quality rules.
        passed_df, rejected_df, dq_metrics = execute_dataset_dq(
            transformed_df,
            dq_rules
        )

        print(
            "DQ execution completed. "
            "Continuing without DataFrame caching."
        )

        # Calculate DQ metrics.
        input_count = transformed_df.count()
        rejected_count = rejected_df.count()

        reject_ratio = (
            rejected_count / input_count
            if input_count
            else 0.0
        )

        # Persist rejected records in quarantine.
        quarantine_count = write_quarantine(
            rejected_df,
            paths["quarantine"],
            DATASET_NAME
        )

        print(
            f"DQ input={input_count}; "
            f"rejected={rejected_count}; "
            f"ratio={reject_ratio:.2%}"
        )

        # Stop if rejection ratio exceeds the allowed threshold.
        if reject_ratio > MAX_REJECT_RATIO:
            raise ValueError(
                f"Reject ratio {reject_ratio:.2%} exceeds threshold "
                f"{MAX_REJECT_RATIO:.2%}; "
                "Silver target and watermark were not advanced."
            )

        # Stop if no valid records remain.
        if passed_df.limit(1).count() == 0:
            print(
                "No valid records remain after DQ. "
                "Silver target and watermark unchanged."
            )
            return None

        # Remove internal processing columns.
        valid_df = passed_df.drop(
            *[
                c for c in ("_cast_errors",)
                if c in passed_df.columns
            ]
        )

        # SCD1 deduplication.
        if (
            int(config["scd_type"]) == 1
            and config.get("deduplication_enabled", True)
        ):
            valid_df = deduplicate_records(
                valid_df,
                config["business_key_columns"],
                config.get("sequence_column")
            )

        # Prepare the Silver source.
        source_df = prepare_silver_source(
            valid_df,
            config["business_key_columns"]
        )

        # Execute the appropriate SCD strategy.
        if int(config["scd_type"]) == 1:

            merge_input_count = merge_scd1(
                source_df,
                paths["silver"],
                config["business_key_columns"],
                reprocess=REPROCESS
            )

        else:

            merge_input_count = merge_scd2_incremental(
                source_df,
                paths["silver"],
                config
            )

        # Validate the final Silver table.
        final_silver_df = validate_silver(
            paths["silver"],
            config
        )

        # Advance watermark only after successful Silver write
        # and final validation.
        processed_max_timestamp = (
            bronze_df
            .agg(
                F.max("ingestion_timestamp").alias(
                    "max_ingestion_timestamp"
                )
            )
            .first()["max_ingestion_timestamp"]
        )

        update_watermark(processed_max_timestamp)

        # Final execution summary.
        print("=" * 72)
        print("SILVER PIPELINE SUCCEEDED")
        print(f"Dataset                 : {DATASET_NAME}")
        print(f"Schema version          : {schema_version}")
        print(f"Source files            : {file_count}")
        print(f"Input rows              : {input_count}")
        print(f"Rejected rows           : {rejected_count}")
        print(f"Quarantine upserts      : {quarantine_count}")
        print(f"Silver write rows       : {merge_input_count}")
        print(f"Silver total rows       : {final_silver_df.count()}")
        print(
            f"Watermark lookback      : "
            f"{WATERMARK_LOOKBACK_MINUTES} minutes"
        )
        print(f"DQ rule metrics         : {dq_metrics}")
        print("=" * 72)

        return final_silver_df

    except Exception as exc:

        print(
            f"SILVER PIPELINE FAILED: "
            f"{type(exc).__name__}: {exc}"
        )

        raise


In [0]:
# CELL 22 — RUN PIPELINE
# Before running:
# 1. Confirm Notebook 05 and Notebook 06 %run paths resolve.
# 2. Dataset config must be active and APPROVED.
# 3. DQ rules are loaded only when active AND APPROVED.
# 4. Keep require_ledger=false until the commit-ledger contract is implemented.
# 5. Initial watermark is NULL, so all Bronze history is selected.
# 6. SCD2 initial bootstrap requires at most one event per key in that history.

silver_result_df = run_silver_pipeline()
if silver_result_df is not None:
    display(silver_result_df)


Starting Silver pipeline | dataset=google_ads_campaigns | run=e0c8acb8-e1be-4a14-94b6-46114f43c154
silver_dataset_config: EXISTS | abfss://customer-sales-datalake@nestlesalesdatalake.dfs.core.windows.net/metadata/silver_dataset_config/
silver_column_transform_config: EXISTS | abfss://customer-sales-datalake@nestlesalesdatalake.dfs.core.windows.net/metadata/silver_column_transform_config/
silver_dq_rules_config: EXISTS | abfss://customer-sales-datalake@nestlesalesdatalake.dfs.core.windows.net/metadata/silver_dq_rules_config/
schema_config: EXISTS | abfss://customer-sales-datalake@nestlesalesdatalake.dfs.core.windows.net/metadata/schema_config/
Previous successful watermark: None
Bronze selection: INITIAL LOAD — all available Bronze records
Selected Bronze rows=100; source files=1
Schema drift check: {'mode': 'STRICT', 'missing': [], 'added': [], 'has_drift': False}
Applying 8 transformation rules
Starting Silver transformations: 8 rules.
Applying TRIM on 'business_brand' with target 'bu

customer_id,campaign_id,campaign_resource_name,campaign_name,campaign_status,advertising_channel_type,advertising_channel_sub_type,campaign_budget_resource_name,bidding_strategy_type,start_date,end_date,campaign_primary_status,serving_status,target_country,business_brand,product_category,business_objective,source_extract_timestamp,ingestion_timestamp,source_system,source_file,run_id,batch_id,created_at,updated_at,_silver_row_hash,_scd2_version_id,effective_from,effective_to,is_current
1234500000,987650008,customers/1234500000/campaigns/987650008,IN_Purina_Video_008,PAUSED,VIDEO,SMART,customers/1234500000/campaignBudgets/555008,TARGET_CPA,2026-01-25,null,LIMITED,UNKNOWN,US,Purina,Pet Care,Conversions,2026-10-29T03:38:00Z,2026-10-30T07:54:58.051Z,google_ads,campaigns_20261029.json,029dd48f-f521-4626-ab84-1868055c59a7,26c28e16-6aaa-40ea-8a45-ccfc11aef9d3,2026-11-01T11:22:58.088Z,2026-11-01T11:22:58.088Z,d06195589584cf7ba0d30d18afffdbd56f2e951fc7eb746c9bb4a9c7c8a8003b,c93a2ff489f1c399ed2b21f46dbda18e5f24b5fd0f5ab7b036c3f7d2cb505090,2026-10-30T07:54:58.051Z,null,true
1234500000,987650010,customers/1234500000/campaigns/987650010,IN_KitKat_Performance_Max_010,ENABLED,PERFORMANCE_MAX,VIDEO_ACTION,customers/1234500000/campaignBudgets/555010,TARGET_ROAS,2026-01-31,2026-05-11,PAUSED,NOT_SERVING,AU,KitKat,Chocolate,Sales,2026-10-29T03:40:00Z,2026-10-30T07:54:58.051Z,google_ads,campaigns_20261029.json,029dd48f-f521-4626-ab84-1868055c59a7,26c28e16-6aaa-40ea-8a45-ccfc11aef9d3,2026-11-01T11:22:58.088Z,2026-11-01T11:22:58.088Z,99ab00a6dbb4fa8a50c0b3d8cb9738c068b32c22939f4a68d7db4e1231b7ad49,9a512b235cba938e14510b3d0903040f4bc4477ebf9b4de1dafa4eb4c72d91bd,2026-10-30T07:54:58.051Z,null,true
1234500002,987650024,customers/1234500002/campaigns/987650024,IN_Purina_Shopping_024,REMOVED,SHOPPING,null,customers/1234500002/campaignBudgets/555024,MANUAL_CPC,2026-03-14,null,ELIGIBLE,SERVING,CA,Purina,Pet Care,Conversions,2026-10-29T03:54:00Z,2026-10-30T07:54:58.051Z,google_ads,campaigns_20261029.json,029dd48f-f521-4626-ab84-1868055c59a7,26c28e16-6aaa-40ea-8a45-ccfc11aef9d3,2026-11-01T11:22:58.088Z,2026-11-01T11:22:58.088Z,c583cc0c63e4154befd7bbb4ebaa1362bd7e03084254edf09108ebb3ddd7c541,5cc190c0be27ea989d0839e3411459f7af02edc642750f1c22bac211aa1e5d17,2026-10-30T07:54:58.051Z,null,true
1234500003,987650032,customers/1234500003/campaigns/987650032,IN_Purina_Display_032,PAUSED,DISPLAY,SMART,customers/1234500003/campaignBudgets/555032,MAXIMIZE_CLICKS,2026-04-07,null,LIMITED,UNKNOWN,US,Purina,Pet Care,Conversions,2026-10-29T04:02:00Z,2026-10-30T07:54:58.051Z,google_ads,campaigns_20261029.json,029dd48f-f521-4626-ab84-1868055c59a7,26c28e16-6aaa-40ea-8a45-ccfc11aef9d3,2026-11-01T11:22:58.088Z,2026-11-01T11:22:58.088Z,210b415652ab8b761cb3954c620c36c00e0f7f66be901a36052bc361d06dc1cf,30e2ad2f33258d3d2c5001c32cf5b3931926a5c03a559f4f212d5987e7cfef49,2026-10-30T07:54:58.051Z,null,true
1234500003,987650037,customers/1234500003/campaigns/987650037,IN_Cerelac_Display_037,ENABLED,DISPLAY,VIDEO_ACTION,customers/1234500003/campaignBudgets/555037,MAXIMIZE_CLICKS,2026-04-22,2026-08-27,PAUSED,NOT_SERVING,IN,Cerelac,Nutrition,Consideration,2026-10-29T04:07:00Z,2026-10-30T07:54:58.051Z,google_ads,campaigns_20261029.json,029dd48f-f521-4626-ab84-1868055c59a7,26c28e16-6aaa-40ea-8a45-ccfc11aef9d3,2026-11-01T11:22:58.088Z,2026-11-01T11:22:58.088Z,118a34cc46d61c00f1cd38ff8db3d71159569e36fd0041ebdfada30a7fdf5336,8c0c4605c164124fea741a25154bf064013c1062ec9dded3dc6bb3f2cf41de40,2026-10-30T07:54:58.051Z,null,true
1234500005,987650052,customers/1234500005/campaigns/987650052,IN_Milo_Display_052,ENABLED,DISPLAY,VIDEO_ACTION,customers/1234500005/campaignBudgets/555052,MAXIMIZE_CLICKS,2026-06-06,null,PAUSED,NOT_SERVING,AU,Milo,Malted Beverages,Reach,2026-10-29T04:22:00Z,2026-10-30T07:54:58.051Z,google_ads,campaigns_20261029.json,029dd48f-f521-4626-ab84-1868055c59a7,26c28e16-6aaa-40ea-8a45-ccfc11aef9d3,2026-11-01T11:22:58.088Z,2026-11-01T11:22:58.088Z,a931949568e97f58d5f2e554e78d1227ec97f35cbea

In [0]:
# CELL 23 — OPTIONAL DELTA OPERATION METRICS

def last_operation_metrics(delta_path):
    if not DeltaTable.isDeltaTable(spark, delta_path):
        print(f"No Delta table found at {delta_path}")
        return {}
    row = (DeltaTable.forPath(spark, delta_path).history(1)
           .select("operation", "operationMetrics").first())
    return {"operation": row["operation"], "operationMetrics": row["operationMetrics"]}

# Optional after a successful run:
# print(last_operation_metrics(dataset_paths(DATASET_NAME)["silver"]))


In [0]:
# CELL 24 — READ-ONLY CONFIGURATION / WATERMARK SUMMARY

print("Silver notebook configuration summary")
print(f"Dataset               : {DATASET_NAME}")
print(f"Pipeline              : {PIPELINE_NAME}")
print(f"Bronze path           : {dataset_paths(DATASET_NAME)['bronze']}")
print(f"Silver path           : {dataset_paths(DATASET_NAME)['silver']}")
print(f"Quarantine path       : {dataset_paths(DATASET_NAME)['quarantine']}")
print(f"Watermark path        : {WATERMARK_CONTROL_PATH}")
print(f"Lookback minutes      : {WATERMARK_LOOKBACK_MINUTES}")
print(f"Last successful mark  : {read_last_watermark()}")


Silver notebook configuration summary
Dataset               : google_ads_campaigns
Pipeline              : google_ads_campaigns_silver
Bronze path           : abfss://customer-sales-datalake@nestlesalesdatalake.dfs.core.windows.net/bronze/google_ads_campaigns/
Silver path           : abfss://customer-sales-datalake@nestlesalesdatalake.dfs.core.windows.net/silver/google_ads_campaigns/
Quarantine path       : abfss://customer-sales-datalake@nestlesalesdatalake.dfs.core.windows.net/quarantine/silver/google_ads_campaigns/
Watermark path        : abfss://customer-sales-datalake@nestlesalesdatalake.dfs.core.windows.net/metadata/pipeline_watermark/
Lookback minutes      : 5
Last successful mark  : 2026-10-30 07:54:58.051651


In [0]:
from delta.tables import DeltaTable
from pyspark.sql import functions as F

SILVER_PATH = (
    "abfss://customer-sales-datalake@nestlesalesdatalake.dfs.core.windows.net/"
    "silver/google_ads_campaigns/"
)

if DeltaTable.isDeltaTable(spark, SILVER_PATH):

    silver_df = spark.read.format("delta").load(SILVER_PATH)

    print("Silver Delta table EXISTS")
    print("Total records:", silver_df.count())

    display(silver_df.limit(10))

else:
    print("Silver Delta table DOES NOT EXIST")

Silver Delta table EXISTS
Total records: 100


customer_id,campaign_id,campaign_resource_name,campaign_name,campaign_status,advertising_channel_type,advertising_channel_sub_type,campaign_budget_resource_name,bidding_strategy_type,start_date,end_date,campaign_primary_status,serving_status,target_country,business_brand,product_category,business_objective,source_extract_timestamp,ingestion_timestamp,source_system,source_file,run_id,batch_id,created_at,updated_at,_silver_row_hash,_scd2_version_id,effective_from,effective_to,is_current
1234500000,987650008,customers/1234500000/campaigns/987650008,IN_Purina_Video_008,PAUSED,VIDEO,SMART,customers/1234500000/campaignBudgets/555008,TARGET_CPA,2026-01-25,null,LIMITED,UNKNOWN,US,Purina,Pet Care,Conversions,2026-10-29T03:38:00Z,2026-10-30T07:54:58.051Z,google_ads,campaigns_20261029.json,029dd48f-f521-4626-ab84-1868055c59a7,26c28e16-6aaa-40ea-8a45-ccfc11aef9d3,2026-11-01T11:22:58.088Z,2026-11-01T11:22:58.088Z,d06195589584cf7ba0d30d18afffdbd56f2e951fc7eb746c9bb4a9c7c8a8003b,c93a2ff489f1c399ed2b21f46dbda18e5f24b5fd0f5ab7b036c3f7d2cb505090,2026-10-30T07:54:58.051Z,null,true
1234500000,987650010,customers/1234500000/campaigns/987650010,IN_KitKat_Performance_Max_010,ENABLED,PERFORMANCE_MAX,VIDEO_ACTION,customers/1234500000/campaignBudgets/555010,TARGET_ROAS,2026-01-31,2026-05-11,PAUSED,NOT_SERVING,AU,KitKat,Chocolate,Sales,2026-10-29T03:40:00Z,2026-10-30T07:54:58.051Z,google_ads,campaigns_20261029.json,029dd48f-f521-4626-ab84-1868055c59a7,26c28e16-6aaa-40ea-8a45-ccfc11aef9d3,2026-11-01T11:22:58.088Z,2026-11-01T11:22:58.088Z,99ab00a6dbb4fa8a50c0b3d8cb9738c068b32c22939f4a68d7db4e1231b7ad49,9a512b235cba938e14510b3d0903040f4bc4477ebf9b4de1dafa4eb4c72d91bd,2026-10-30T07:54:58.051Z,null,true
1234500002,987650024,customers/1234500002/campaigns/987650024,IN_Purina_Shopping_024,REMOVED,SHOPPING,null,customers/1234500002/campaignBudgets/555024,MANUAL_CPC,2026-03-14,null,ELIGIBLE,SERVING,CA,Purina,Pet Care,Conversions,2026-10-29T03:54:00Z,2026-10-30T07:54:58.051Z,google_ads,campaigns_20261029.json,029dd48f-f521-4626-ab84-1868055c59a7,26c28e16-6aaa-40ea-8a45-ccfc11aef9d3,2026-11-01T11:22:58.088Z,2026-11-01T11:22:58.088Z,c583cc0c63e4154befd7bbb4ebaa1362bd7e03084254edf09108ebb3ddd7c541,5cc190c0be27ea989d0839e3411459f7af02edc642750f1c22bac211aa1e5d17,2026-10-30T07:54:58.051Z,null,true
1234500003,987650032,customers/1234500003/campaigns/987650032,IN_Purina_Display_032,PAUSED,DISPLAY,SMART,customers/1234500003/campaignBudgets/555032,MAXIMIZE_CLICKS,2026-04-07,null,LIMITED,UNKNOWN,US,Purina,Pet Care,Conversions,2026-10-29T04:02:00Z,2026-10-30T07:54:58.051Z,google_ads,campaigns_20261029.json,029dd48f-f521-4626-ab84-1868055c59a7,26c28e16-6aaa-40ea-8a45-ccfc11aef9d3,2026-11-01T11:22:58.088Z,2026-11-01T11:22:58.088Z,210b415652ab8b761cb3954c620c36c00e0f7f66be901a36052bc361d06dc1cf,30e2ad2f33258d3d2c5001c32cf5b3931926a5c03a559f4f212d5987e7cfef49,2026-10-30T07:54:58.051Z,null,true
1234500003,987650037,customers/1234500003/campaigns/987650037,IN_Cerelac_Display_037,ENABLED,DISPLAY,VIDEO_ACTION,customers/1234500003/campaignBudgets/555037,MAXIMIZE_CLICKS,2026-04-22,2026-08-27,PAUSED,NOT_SERVING,IN,Cerelac,Nutrition,Consideration,2026-10-29T04:07:00Z,2026-10-30T07:54:58.051Z,google_ads,campaigns_20261029.json,029dd48f-f521-4626-ab84-1868055c59a7,26c28e16-6aaa-40ea-8a45-ccfc11aef9d3,2026-11-01T11:22:58.088Z,2026-11-01T11:22:58.088Z,118a34cc46d61c00f1cd38ff8db3d71159569e36fd0041ebdfada30a7fdf5336,8c0c4605c164124fea741a25154bf064013c1062ec9dded3dc6bb3f2cf41de40,2026-10-30T07:54:58.051Z,null,true
1234500005,987650052,customers/1234500005/campaigns/987650052,IN_Milo_Display_052,ENABLED,DISPLAY,VIDEO_ACTION,customers/1234500005/campaignBudgets/555052,MAXIMIZE_CLICKS,2026-06-06,null,PAUSED,NOT_SERVING,AU,Milo,Malted Beverages,Reach,2026-10-29T04:22:00Z,2026-10-30T07:54:58.051Z,google_ads,campaigns_20261029.json,029dd48f-f521-4626-ab84-1868055c59a7,26c28e16-6aaa-40ea-8a45-ccfc11aef9d3,2026-11-01T11:22:58.088Z,2026-11-01T11:22:58.088Z,a931949568e97f58d5f2e554e78d1227ec97f35cbea